## Few-shot 프롬프트 (최신 권장 방식)

> 원본: CH02 `02-FewShotTemplates.ipynb`  
> 기준: LangChain 1.x / langchain-core 1.2+ (2026년 9월)

In [2]:
# 필요 패키지 설치 (최초 1회)
# %pip install -qU langchain langchain-openai langsmith python-dotenv

### 🔄 변경 사항: 환경 설정
- `langchain_teddynote.logging.langsmith()` → LangSmith 공식 환경변수 `LANGSMITH_TRACING`, `LANGSMITH_PROJECT` 직접 설정  
  (써드파티 래퍼 없이 동일하게 동작합니다. `.env` 파일에 넣어 두어도 됩니다.)

In [3]:
import os
from dotenv import load_dotenv

# .env 에 OPENAI_API_KEY, LANGSMITH_API_KEY 를 넣어 두세요.
load_dotenv()

# LangSmith 추적 설정
# (langchain_teddynote.logging.langsmith() 대신, LangSmith 공식 환경변수를 직접 설정)
os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_PROJECT"] = "CH02-Prompt"

## FewShotPromptTemplate

### 🔄 변경 사항
- `ChatOpenAI(temperature=0, model_name="gpt-4.1")` → `init_chat_model(MODEL)` (`model_name` 인자는 구식 별칭이며 `model` 이 표준입니다.)
- `langchain_teddynote.messages.stream_response` → 표준 `for chunk in ....stream(...)` 루프.  
  아래에 같은 역할을 하는 작은 헬퍼 `print_stream` 을 직접 정의해 사용합니다.

In [4]:
from langchain.chat_models import init_chat_model

# 모델은 "provider:model" 문자열 하나로 지정합니다.
# 다른 모델로 바꾸려면 이 한 줄만 수정하면 됩니다. (예: "anthropic:claude-sonnet-4-6")
MODEL = "openai:gpt-5.6-luna"

llm = init_chat_model(MODEL)

In [5]:
def print_stream(stream):
    """stream() 결과를 실시간 출력 (문자열 청크 / 메시지 청크 모두 처리)"""
    for chunk in stream:
        print(chunk if isinstance(chunk, str) else chunk.text, end="", flush=True)
    print()


# 질의내용
question = "대한민국의 수도는 뭐야?"

# 질의
print_stream(llm.stream(question))

대한민국의 수도는 서울특별시입니다.


### 🔄 변경 사항
- `from langchain_core.prompts.few_shot import FewShotPromptTemplate` → `from langchain_core.prompts import FewShotPromptTemplate` (공개 경로로 import)
- `FewShotPromptTemplate` 은 **문자열 프롬프트**를 만듭니다. 채팅 모델에는 아래 `FewShotChatMessagePromptTemplate` 이 더 자연스러운 선택이지만, 개념 학습을 위해 그대로 다룹니다.

In [6]:
from langchain_core.prompts import FewShotPromptTemplate, PromptTemplate
from langchain_core.output_parsers import StrOutputParser


examples = [
    {
        "question": "스티브 잡스와 아인슈타인 중 누가 더 오래 살았나요?",
        "answer": """이 질문에 추가 질문이 필요한가요: 예.
추가 질문: 스티브 잡스는 몇 살에 사망했나요?
중간 답변: 스티브 잡스는 56세에 사망했습니다.
추가 질문: 아인슈타인은 몇 살에 사망했나요?
중간 답변: 아인슈타인은 76세에 사망했습니다.
최종 답변은: 아인슈타인
""",
    },
    {
        "question": "네이버의 창립자는 언제 태어났나요?",
        "answer": """이 질문에 추가 질문이 필요한가요: 예.
추가 질문: 네이버의 창립자는 누구인가요?
중간 답변: 네이버는 이해진에 의해 창립되었습니다.
추가 질문: 이해진은 언제 태어났나요?
중간 답변: 이해진은 1967년 6월 22일에 태어났습니다.
최종 답변은: 1967년 6월 22일
""",
    },
    {
        "question": "율곡 이이의 어머니가 태어난 해의 통치하던 왕은 누구인가요?",
        "answer": """이 질문에 추가 질문이 필요한가요: 예.
추가 질문: 율곡 이이의 어머니는 누구인가요?
중간 답변: 율곡 이이의 어머니는 신사임당입니다.
추가 질문: 신사임당은 언제 태어났나요?
중간 답변: 신사임당은 1504년에 태어났습니다.
추가 질문: 1504년에 조선을 통치한 왕은 누구인가요?
중간 답변: 1504년에 조선을 통치한 왕은 연산군입니다.
최종 답변은: 연산군
""",
    },
    {
        "question": "올드보이와 기생충의 감독이 같은 나라 출신인가요?",
        "answer": """이 질문에 추가 질문이 필요한가요: 예.
추가 질문: 올드보이의 감독은 누구인가요?
중간 답변: 올드보이의 감독은 박찬욱입니다.
추가 질문: 박찬욱은 어느 나라 출신인가요?
중간 답변: 박찬욱은 대한민국 출신입니다.
추가 질문: 기생충의 감독은 누구인가요?
중간 답변: 기생충의 감독은 봉준호입니다.
추가 질문: 봉준호는 어느 나라 출신인가요?
중간 답변: 봉준호는 대한민국 출신입니다.
최종 답변은: 예
""",
    },
]

In [7]:
example_prompt = PromptTemplate.from_template(
    "Question:\n{question}\nAnswer:\n{answer}"
)

print(example_prompt.format(**examples[0]))

Question:
스티브 잡스와 아인슈타인 중 누가 더 오래 살았나요?
Answer:
이 질문에 추가 질문이 필요한가요: 예.
추가 질문: 스티브 잡스는 몇 살에 사망했나요?
중간 답변: 스티브 잡스는 56세에 사망했습니다.
추가 질문: 아인슈타인은 몇 살에 사망했나요?
중간 답변: 아인슈타인은 76세에 사망했습니다.
최종 답변은: 아인슈타인



In [8]:
prompt = FewShotPromptTemplate(
    examples=examples,
    example_prompt=example_prompt,
    suffix="Question:\n{question}\nAnswer:",
    input_variables=["question"],
)

question = "Google이 창립된 연도에 Bill Gates의 나이는 몇 살인가요?"
final_prompt = prompt.format(question=question)
print(final_prompt)

Question:
스티브 잡스와 아인슈타인 중 누가 더 오래 살았나요?
Answer:
이 질문에 추가 질문이 필요한가요: 예.
추가 질문: 스티브 잡스는 몇 살에 사망했나요?
중간 답변: 스티브 잡스는 56세에 사망했습니다.
추가 질문: 아인슈타인은 몇 살에 사망했나요?
중간 답변: 아인슈타인은 76세에 사망했습니다.
최종 답변은: 아인슈타인


Question:
네이버의 창립자는 언제 태어났나요?
Answer:
이 질문에 추가 질문이 필요한가요: 예.
추가 질문: 네이버의 창립자는 누구인가요?
중간 답변: 네이버는 이해진에 의해 창립되었습니다.
추가 질문: 이해진은 언제 태어났나요?
중간 답변: 이해진은 1967년 6월 22일에 태어났습니다.
최종 답변은: 1967년 6월 22일


Question:
율곡 이이의 어머니가 태어난 해의 통치하던 왕은 누구인가요?
Answer:
이 질문에 추가 질문이 필요한가요: 예.
추가 질문: 율곡 이이의 어머니는 누구인가요?
중간 답변: 율곡 이이의 어머니는 신사임당입니다.
추가 질문: 신사임당은 언제 태어났나요?
중간 답변: 신사임당은 1504년에 태어났습니다.
추가 질문: 1504년에 조선을 통치한 왕은 누구인가요?
중간 답변: 1504년에 조선을 통치한 왕은 연산군입니다.
최종 답변은: 연산군


Question:
올드보이와 기생충의 감독이 같은 나라 출신인가요?
Answer:
이 질문에 추가 질문이 필요한가요: 예.
추가 질문: 올드보이의 감독은 누구인가요?
중간 답변: 올드보이의 감독은 박찬욱입니다.
추가 질문: 박찬욱은 어느 나라 출신인가요?
중간 답변: 박찬욱은 대한민국 출신입니다.
추가 질문: 기생충의 감독은 누구인가요?
중간 답변: 기생충의 감독은 봉준호입니다.
추가 질문: 봉준호는 어느 나라 출신인가요?
중간 답변: 봉준호는 대한민국 출신입니다.
최종 답변은: 예


Question:
Google이 창립된 연도에 Bill Gates의 나이는 몇 살인가요?
Answer:


In [9]:
# 결과 출력
print_stream(llm.stream(final_prompt))

이 질문에 추가 질문이 필요한가요: 예.  
추가 질문: Google은 언제 창립되었나요?  
중간 답변: Google은 1998년 9월 4일에 창립되었습니다.  
추가 질문: Bill Gates는 언제 태어났나요?  
중간 답변: Bill Gates는 1955년 10월 28일에 태어났습니다.  
추가 질문: Google이 창립된 날 Bill Gates의 나이는 몇 살인가요?  
중간 답변: Bill Gates는 1998년 9월 4일에 42세였습니다.  
최종 답변은: 42세


In [10]:
# chain 생성
chain = prompt | llm | StrOutputParser()

# 결과 출력
print_stream(
    chain.stream({"question": "Google이 창립된 연도에 Bill Gates의 나이는 몇 살인가요?"})
)

이 질문에 추가 질문이 필요한가요: 예.  
추가 질문: Google은 언제 창립되었나요?  
중간 답변: Google은 1998년에 창립되었습니다.  
추가 질문: Bill Gates는 언제 태어났나요?  
중간 답변: Bill Gates는 1955년 10월 28일에 태어났습니다.  
추가 질문: 1998년에 Bill Gates의 나이는 몇 살인가요?  
중간 답변: 1998년에 Bill Gates는 42세였습니다.  
최종 답변은: 42세


## Example Selector

예제가 많은 경우 프롬프트에 포함할 예제를 선택해야 할 수도 있습니다. Example Selector 는 이 작업을 담당하는 클래스입니다.

### 🔄 변경 사항
- 벡터 저장소: `langchain_chroma.Chroma` → **`langchain_core.vectorstores.InMemoryVectorStore`**  
  예제 선택 정도의 소규모 작업에는 별도 DB 패키지 없이 core 에 내장된 인메모리 저장소로 충분합니다. 또한 Chroma 는 같은 컬렉션명으로 셀을 재실행하면 예제가 **중복 누적**되는 문제가 있었는데, 이것도 사라집니다.  
  (원본의 `chroma = Chroma("example_selector", ...)` 줄은 실제로 사용되지 않는 코드였으므로 제거했습니다.)
- 임베딩 모델을 명시합니다: `OpenAIEmbeddings(model="text-embedding-3-small")`
- **`input_keys=["question"]`** 지정: 검색 시 `question` 만 임베딩하여 비교합니다. 지정하지 않으면 예제의 모든 값(질문+답변)을 이어붙여 임베딩하므로 유사도가 흐려집니다. (아래 "유사도 검색 문제 해결" 절의 핵심이기도 합니다.)
- 참고: 다양성을 고려한 선택이 필요하면 같은 인터페이스의 `MaxMarginalRelevanceExampleSelector` 를 쓰면 됩니다.
- [API 문서](https://reference.langchain.com/python/langchain-core/example_selectors/semantic_similarity/SemanticSimilarityExampleSelector/from_examples)

In [13]:
from langchain_core.example_selectors import SemanticSimilarityExampleSelector
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

example_selector = SemanticSimilarityExampleSelector.from_examples(
    # 선택 가능한 예시 목록
    examples,
    # 의미적 유사성 측정용 임베딩 생성 클래스
    embeddings,
    # 임베딩 저장 및 유사성 검색에 사용할 VectorStore "클래스"
    InMemoryVectorStore,
    # 선택할 예시의 수
    k=1,
    # 유사도 비교에 사용할 키 (question 만 비교)
    input_keys=["question"],
)

question = "Google이 창립된 연도에 Bill Gates의 나이는 몇 살인가요?"

# 입력과 가장 유사한 예시를 선택합니다.
selected_examples = example_selector.select_examples({"question": question})

print(f"입력에 가장 유사한 예시:\n{question}\n")
for example in selected_examples:
    print(f'question:\n{example["question"]}')
    print(f'answer:\n{example["answer"]}')

ImportError: cosine_similarity requires numpy to be installed. Please install numpy with `pip install numpy`.

In [ ]:
prompt = FewShotPromptTemplate(
    example_selector=example_selector,
    example_prompt=example_prompt,
    suffix="Question:\n{question}\nAnswer:",
    input_variables=["question"],
)

question = "Google이 창립된 연도에 Bill Gates의 나이는 몇 살인가요?"
example_selector_prompt = prompt.format(question=question)
print(example_selector_prompt)

이번에는 ExampleSelector 를 사용한 FewShotPromptTemplate 으로 체인을 생성합니다.

In [ ]:
# 체인 생성
chain = prompt | llm | StrOutputParser()

In [ ]:
# 결과 출력
print_stream(
    chain.stream({"question": "Google이 창립된 연도에 Bill Gates의 나이는 몇 살인가요?"})
)

## FewShotChatMessagePromptTemplate

채팅 모델에는 예제를 human/ai 메시지 쌍으로 넣는 이 방식이 권장됩니다.

### 🔄 변경 사항
- 두 번째 예제 목록은 이름을 `chat_examples` 로 바꿔 위의 `examples` 와 섞이지 않게 했습니다.
- 원본은 `from_examples()` 에 Chroma **인스턴스**를 넘겼지만, 이 인자는 **VectorStore 클래스**를 받습니다. `InMemoryVectorStore` 클래스를 넘깁니다.

In [ ]:
chat_examples = [
    {
        "instruction": "당신은 회의록 작성 전문가 입니다. 주어진 정보를 바탕으로 회의록을 작성해 주세요",
        "input": "2023년 12월 25일, XYZ 회사의 마케팅 전략 회의가 오후 3시에 시작되었다. 회의에는 마케팅 팀장인 김수진, 디지털 마케팅 담당자인 박지민, 소셜 미디어 관리자인 이준호가 참석했다. 회의의 주요 목적은 2024년 상반기 마케팅 전략을 수립하고, 새로운 소셜 미디어 캠페인에 대한 아이디어를 논의하는 것이었다. 팀장인 김수진은 최근 시장 동향에 대한 간략한 개요를 제공했으며, 이어서 각 팀원이 자신의 분야에서의 전략적 아이디어를 발표했다.",
        "answer": """
회의록: XYZ 회사 마케팅 전략 회의
일시: 2023년 12월 25일
장소: XYZ 회사 회의실
참석자: 김수진 (마케팅 팀장), 박지민 (디지털 마케팅 담당자), 이준호 (소셜 미디어 관리자)

1. 개회
   - 회의는 김수진 팀장의 개회사로 시작됨.
   - 회의의 목적은 2024년 상반기 마케팅 전략 수립 및 새로운 소셜 미디어 캠페인 아이디어 논의.

2. 시장 동향 개요 (김수진)
   - 김수진 팀장은 최근 시장 동향에 대한 분석을 제시.
   - 소비자 행동 변화와 경쟁사 전략에 대한 통찰 공유.

3. 디지털 마케팅 전략 (박지민)
   - 박지민은 디지털 마케팅 전략에 대해 발표.
   - 온라인 광고와 SEO 최적화 방안에 중점을 둠.

4. 소셜 미디어 캠페인 (이준호)
   - 이준호는 새로운 소셜 미디어 캠페인에 대한 아이디어를 제안.
   - 인플루언서 마케팅과 콘텐츠 전략에 대한 계획을 설명함.

5. 종합 논의
   - 팀원들 간의 아이디어 공유 및 토론.
   - 각 전략에 대한 예산 및 자원 배분에 대해 논의.

6. 마무리
   - 다음 회의 날짜 및 시간 확정.
   - 회의록 정리 및 배포는 박지민 담당.
""",
    },
    {
        "instruction": "당신은 요약 전문가 입니다. 다음 주어진 정보를 바탕으로 내용을 요약해 주세요",
        "input": "이 문서는 '지속 가능한 도시 개발을 위한 전략'에 대한 20페이지 분량의 보고서입니다. 보고서는 지속 가능한 도시 개발의 중요성, 현재 도시화의 문제점, 그리고 도시 개발을 지속 가능하게 만들기 위한 다양한 전략을 포괄적으로 다루고 있습니다. 이 보고서는 또한 성공적인 지속 가능한 도시 개발 사례를 여러 국가에서 소개하고, 이러한 사례들을 통해 얻은 교훈을 요약하고 있습니다.",
        "answer": """
문서 요약: 지속 가능한 도시 개발을 위한 전략 보고서

- 중요성: 지속 가능한 도시 개발이 필수적인 이유와 그에 따른 사회적, 경제적, 환경적 이익을 강조.
- 현 문제점: 현재의 도시화 과정에서 발생하는 주요 문제점들, 예를 들어 환경 오염, 자원 고갈, 불평등 증가 등을 분석.
- 전략: 지속 가능한 도시 개발을 달성하기 위한 다양한 전략 제시. 이에는 친환경 건축, 대중교통 개선, 에너지 효율성 증대, 지역사회 참여 강화 등이 포함됨.
- 사례 연구: 전 세계 여러 도시의 성공적인 지속 가능한 개발 사례를 소개. 예를 들어, 덴마크의 코펜하겐, 일본의 요코하마 등의 사례를 통해 실현 가능한 전략들을 설명.
- 교훈: 이러한 사례들에서 얻은 주요 교훈을 요약. 강조된 교훈에는 다각적 접근의 중요성, 지역사회와의 협력, 장기적 계획의 필요성 등이 포함됨.

이 보고서는 지속 가능한 도시 개발이 어떻게 현실적이고 효과적인 형태로 이루어질 수 있는지에 대한 심도 있는 분석을 제공합니다.
""",
    },
    {
        "instruction": "당신은 문장 교정 전문가 입니다. 다음 주어진 문장을 교정해 주세요",
        "input": "우리 회사는 새로운 마케팅 전략을 도입하려고 한다. 이를 통해 고객과의 소통이 더 효과적이 될 것이다.",
        "answer": "본 회사는 새로운 마케팅 전략을 도입함으로써, 고객과의 소통을 보다 효과적으로 개선할 수 있을 것으로 기대된다.",
    },
]

In [ ]:
from langchain_core.prompts import ChatPromptTemplate, FewShotChatMessagePromptTemplate

example_prompt = ChatPromptTemplate(
    [
        ("human", "{instruction}:\n{input}"),
        ("ai", "{answer}"),
    ]
)

# input_keys 를 지정하지 않은 "기본" 선택기 (원본과 동일한 동작)
example_selector = SemanticSimilarityExampleSelector.from_examples(
    chat_examples,
    embeddings,
    InMemoryVectorStore,
    k=1,
)

few_shot_prompt = FewShotChatMessagePromptTemplate(
    example_selector=example_selector,
    example_prompt=example_prompt,
)

fewshot 예제와 example selector를 사용하여 유사한 예제 1개를 선택합니다.

In [ ]:
question = {
    "instruction": "회의록을 작성해 주세요",
    "input": "2023년 12월 26일, ABC 기술 회사의 제품 개발 팀은 새로운 모바일 애플리케이션 프로젝트에 대한 주간 진행 상황 회의를 가졌다. 이 회의에는 프로젝트 매니저인 최현수, 주요 개발자인 황지연, UI/UX 디자이너인 김태영이 참석했다. 회의의 주요 목적은 프로젝트의 현재 진행 상황을 검토하고, 다가오는 마일스톤에 대한 계획을 수립하는 것이었다. 각 팀원은 자신의 작업 영역에 대한 업데이트를 제공했고, 팀은 다음 주까지의 목표를 설정했다.",
}

example_selector.select_examples(question)

In [ ]:
final_prompt = ChatPromptTemplate(
    [
        ("system", "You are a helpful assistant."),
        few_shot_prompt,
        ("human", "{instruction}\n{input}"),
    ]
)

In [ ]:
# chain 생성
chain = final_prompt | llm | StrOutputParser()

# 실행 및 결과 출력
print_stream(chain.stream(question))

### Example Selector 의 유사도 검색 문제 해결

기본 선택기는 예제의 **모든 값**(`instruction` + `input` + `answer`)을 이어붙여 임베딩하고, 질의도 들어온 **모든 값**으로 임베딩합니다. 그래서 `instruction` 만으로 검색하면 제대로된 유사도 결과가 나오지 않습니다.

아래는 잘못 검색될 수 있는 예시입니다.

In [ ]:
example_selector.select_examples({"instruction": "회의록을 작성해 주세요"})

In [ ]:
# 커스텀 하지 않은 기본 예제 선택기를 사용했을 때 결과
example_selector.select_examples({"instruction": "다음 문장을 요약해 주세요"})

### 🔄 변경 사항: 커스텀 클래스 없이 `input_keys` 로 해결
- 원본은 이 문제를 위해 `langchain_teddynote.prompts.CustomExampleSelector` 라는 별도 클래스를 사용했습니다.
- `SemanticSimilarityExampleSelector` 에 **내장된 `input_keys` 파라미터**로 같은 문제를 해결할 수 있습니다.  
  `input_keys=["instruction"]` 을 주면 예제 저장과 질의 모두 `instruction` 값만 임베딩합니다. (선택된 예제는 여전히 전체 키를 반환)
- 주의: `input_keys` 에 지정한 키는 질의 dict 에 반드시 있어야 합니다.

In [ ]:
# instruction 만으로 유사도를 비교하는 예제 선택기
instruction_selector = SemanticSimilarityExampleSelector.from_examples(
    chat_examples,
    embeddings,
    InMemoryVectorStore,
    k=1,
    input_keys=["instruction"],
)

# 결과 확인
instruction_selector.select_examples({"instruction": "다음 문장을 회의록 작성해 주세요"})

In [ ]:
instruction_selector.select_examples({"instruction": "다음 문장을 요약해 주세요"})

In [ ]:
custom_fewshot_prompt = FewShotChatMessagePromptTemplate(
    example_selector=instruction_selector,  # instruction 기반 예제 선택기 사용
    example_prompt=example_prompt,
)

custom_prompt = ChatPromptTemplate(
    [
        ("system", "You are a helpful assistant."),
        custom_fewshot_prompt,
        ("human", "{instruction}\n{input}"),
    ]
)

In [ ]:
# chain 을 생성합니다.
chain = custom_prompt | llm | StrOutputParser()

In [ ]:
question = {
    "instruction": "회의록을 작성해 주세요",
    "input": "2023년 12월 26일, ABC 기술 회사의 제품 개발 팀은 새로운 모바일 애플리케이션 프로젝트에 대한 주간 진행 상황 회의를 가졌다. 이 회의에는 프로젝트 매니저인 최현수, 주요 개발자인 황지연, UI/UX 디자이너인 김태영이 참석했다. 회의의 주요 목적은 프로젝트의 현재 진행 상황을 검토하고, 다가오는 마일스톤에 대한 계획을 수립하는 것이었다. 각 팀원은 자신의 작업 영역에 대한 업데이트를 제공했고, 팀은 다음 주까지의 목표를 설정했다.",
}

# 실행 및 결과 출력
print_stream(chain.stream(question))

In [ ]:
question = {
    "instruction": "문서를 요약해 주세요",
    "input": "이 문서는 '2023년 글로벌 경제 전망'에 관한 30페이지에 달하는 상세한 보고서입니다. 보고서는 세계 경제의 현재 상태, 주요 국가들의 경제 성장률, 글로벌 무역 동향, 그리고 다가오는 해에 대한 경제 예측을 다룹니다. 이 보고서는 또한 다양한 경제적, 정치적, 환경적 요인들이 세계 경제에 미칠 영향을 분석하고 있습니다.",
}

# 실행 및 결과 출력
print_stream(chain.stream(question))

In [ ]:
question = {
    "instruction": "문장을 교정해 주세요",
    "input": "회사는 올해 매출이 증가할 것으로 예상한다. 새로운 전략이 잘 작동하고 있다.",
}

# 실행 및 결과 출력
print_stream(chain.stream(question))